[Guide Index](README.md)


# 03.01 SPARQL

`StarLayerGraph` and `StarLayerDataset` send SPARQL 1.2 to Oxigraph as written. This guide covers running a query and reading its result, prefixes, `DESCRIBE`, writing results and SPARQL Update. The parts of a query that are specific to SPARQL 1.2 are in [03.02](03.02-sparql-12-queries.ipynb), and preparing, extending and checking queries in [03.03](03.03-preparing-extending-checking-queries.ipynb).


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.


In [1]:
from starlayer.graph import StarLayerGraph

EX = "http://example.org/"
RDF = "http://www.w3.org/1999/02/22-rdf-syntax-ns#"

## 03.01.1 Running a query

`query()` runs a SPARQL query on the graph and returns a `Result`; what it holds depends on the query form. 

- **SELETCT** returns a series of rows.  A row gives a variable as an attribute (`row.p`) or a key (`row["p"]`). `result.bindings` gives the same rows as a list of dicts, one per row, each mapping a variable name to its term (a variable left unbound in a row is missing from its dict).

- **ASK** returns a boolean (true/false).  

- **CONSTRUCT** and **DESCRIBE** (section 03.01.3 below) returns a set of triples and a `StarLayerGraph`.



In [2]:
g_team = StarLayerGraph()
# the graph keeps the prefixes the data declares, so queries can use ex: without a PREFIX line (see section 03.01.2)
g_team.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:teamA ex:name "ann", "al" .
    ex:teamB ex:name "bo" .
""", format="turtle")

# SELECT: query() returns a Result; iterate it for the rows
result = g_team.query("SELECT ?team ?n WHERE { ?team ex:name ?n } ORDER BY ?n")
print("Select Rows:")
for row in result:
    print(row.team.value, row["n"].value)

print("\nBindings:")
print(result.bindings)

# ASK: the Result is true or false
result = g_team.query("ASK { ?s ex:name 'bo' }")
print("\nAsk Response:")
print(bool(result))

# CONSTRUCT: .graph is a new StarLayerGraph of the triples the template built
result = g_team.query("CONSTRUCT { ?t ex:called ?n } WHERE { ?t ex:name ?n }")
built = result.graph
print("\nConstruct Triples: ")
print(built.serialize(format="turtle"))


Select Rows:
http://example.org/teamA al
http://example.org/teamA ann
http://example.org/teamB bo

Bindings:
[{'team': <NamedNode value=http://example.org/teamA>, 'n': <Literal value=al datatype=<NamedNode value=http://www.w3.org/2001/XMLSchema#string>>}, {'team': <NamedNode value=http://example.org/teamA>, 'n': <Literal value=ann datatype=<NamedNode value=http://www.w3.org/2001/XMLSchema#string>>}, {'team': <NamedNode value=http://example.org/teamB>, 'n': <Literal value=bo datatype=<NamedNode value=http://www.w3.org/2001/XMLSchema#string>>}]

Ask Response:
True

Construct Triples: 
<http://example.org/teamA> <http://example.org/called> "ann" , "al" .
<http://example.org/teamB> <http://example.org/called> "bo" .



## 03.01.2 Prefixes

A graph keeps a set of prefixes, a dictionary from each prefix to its namespace. It uses them in the queries it runs.  A query which uses the prefixes that are bound to the graph do not need to delcare its own prefixes.  When the graph has not defined the prefixes (or the prefixes in the query differ from those used in the graph):

- a query can declare its own with `PREFIX` line in the query text.
- the `query()` and `update()` functions can provide a `initNs` paramater to delcare prefixes.   


In [3]:
# A new graph has no prefixes.
g_prefixes = StarLayerGraph()
print("new graph:", g_prefixes.prefixes)

# parse() keeps the prefixes that the data declares.
g_prefixes.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:ann ex:knows ex:bo .
""", format="turtle")
print("after parse:", g_prefixes.prefixes)

# query() uses them, so the text needs no PREFIX line.
result = g_prefixes.query("SELECT ?s WHERE { ?s ex:knows ex:bo }")
print("graph prefixes:", [row.s.value for row in result])

# initNs adds a prefix for this call only.
result = g_prefixes.query("SELECT ?s WHERE { ?s ex:knows p:bo }", initNs={"p": EX})
print("initNs:", [row.s.value for row in result])

# A PREFIX line in the text overrides the graph's prefix of the same name. Here ex: means another namespace, so nothing matches.
result = g_prefixes.query("PREFIX ex: <http://other.example/> SELECT ?s WHERE { ?s ex:knows ?o }")
print("PREFIX line:", [row.s.value for row in result])




new graph: {}
after parse: {'ex': 'http://example.org/'}
graph prefixes: ['http://example.org/ann']
initNs: ['http://example.org/ann']
PREFIX line: []


## 03.01.3 Describing a resource

`DESCRIBE` returns the triples that describe a resource (or resources). 

CONSTRUCT builds triples from a template provided in the query. DESCRIBE, as defined in the SPARQL 1.2 specification, has no pre-defined template.  Oxigraph returns the triples that have the resource as their subject. 

`DESCRIBE` takes an IRI or variables bound by a `WHERE` clause. The result is read like a CONSTRUCT result: `result.graph` or `result.triples`.

In [4]:
# DESCRIBE an IRI (g_team is the graph from section 03.01.1)
result = g_team.query("DESCRIBE ex:teamA")
print("Describe an IRI:")
print(result.graph.serialize(format="turtle"))

# DESCRIBE a variable bound by WHERE
result = g_team.query("DESCRIBE ?team WHERE { ?team ex:name 'bo' }")
print("Describe a variable:")
print(result.graph.serialize(format="turtle"))


Describe an IRI:
<http://example.org/teamA> <http://example.org/name> "ann" , "al" .

Describe a variable:
<http://example.org/teamB> <http://example.org/name> "bo" .



## 03.01.4 Writing query results

`Result.serialize(format=None)` writes a result.

A SELECT or ASK result is written as SPARQL Query Results: `"json"` (the default), `"xml"`, `"csv"` or `"tsv"`. The format can be a name or a media type such as `"text/csv"`. A CONSTRUCT or DESCRIBE result (section 03.01.3) is written as RDF, as `StarLayerGraph.serialize` does (Turtle by default), so it takes an RDF format such as `"nt"` or `"turtle"`. A CONSTRUCT or DESCRIBE result is a new graph with no prefixes of its own, so Turtle writes full IRIs unless you pass the prefixes of the graph that was queried: `serialize(format="turtle", prefixes=graph.prefixes)`. A format of the wrong kind, such as `"csv"` for a CONSTRUCT, raises a `ValueError`.

In [5]:
# A small graph to query, so this section runs on its own.
g_team = StarLayerGraph()
g_team.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:teamA ex:name "ann", "al" .
    ex:teamB ex:name "bo" .
""", format="turtle")

# SELECT results as csv and tsv
result = g_team.query("SELECT ?team ?n WHERE { ?team ex:name ?n } ORDER BY ?n")
print("csv format")
print(result.serialize(format="csv"))
print("tsv format")
print(result.serialize(format="tsv"))
print("json format, the default for a SELECT")
print(result.serialize())
print("xml format")
print(result.serialize(format="xml"))

# ASK results are written the same way, as json unless another format is given
print("ASK, json format")
print(g_team.query("ASK { ?s ex:name 'bo' }").serialize())

# CONSTRUCT results are written as RDF. N-Triples does not put the lines in a set order,
# so they are sorted here to give the same output each time.
constructed = g_team.query("CONSTRUCT { ?t ex:called ?n } WHERE { ?t ex:name ?n }")
print("\nn-triples format")
print("\n".join(sorted(constructed.serialize(format="nt").splitlines())))

# The result graph has no prefixes of its own, so Turtle writes full IRIs. Pass the prefixes of the graph that was queried.
# pretty=True groups the statements by subject, in a set order.
print("\nturtle format, with the prefixes of the graph that was queried")
print(constructed.serialize(format="turtle", prefixes=g_team.prefixes, pretty=True))


csv format
team,n
http://example.org/teamA,al
http://example.org/teamA,ann
http://example.org/teamB,bo

tsv format
?team	?n
<http://example.org/teamA>	"al"
<http://example.org/teamA>	"ann"
<http://example.org/teamB>	"bo"

json format, the default for a SELECT
{"head": {"vars": ["team", "n"]}, "results": {"bindings": [{"team": {"type": "uri", "value": "http://example.org/teamA"}, "n": {"type": "literal", "value": "al"}}, {"team": {"type": "uri", "value": "http://example.org/teamA"}, "n": {"type": "literal", "value": "ann"}}, {"team": {"type": "uri", "value": "http://example.org/teamB"}, "n": {"type": "literal", "value": "bo"}}]}}
xml format
<?xml version="1.0"?><sparql xmlns="http://www.w3.org/2005/sparql-results#"><head><variable name="team"/><variable name="n"/></head><results><result><binding name="team"><uri>http://example.org/teamA</uri></binding><binding name="n"><literal>al</literal></binding></result><result><binding name="team"><uri>http://example.org/teamA</uri></binding><bind

## 03.01.5 SPARQL 1.2 Update

`update()` runs SPARQL Update on a graph, including `INSERT DATA` and `DELETE DATA` with SPARQL 1.2 syntax and triple-term patterns. 

In [6]:
# an empty graph that knows the prefixes ex and rdf (EX and RDF, set in the first cell).
g_update = StarLayerGraph(prefixes={"ex": EX, "rdf": RDF})

# INSERT DATA with a triple term. The base triple of the triple term is not asserted.
g_update.update("""
    INSERT DATA { ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> }
""")
print("triples:", len(g_update))
print()
print(g_update.serialize(format="turtle"))

triples: 1

@version "1.2" .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex: <http://example.org/> .
ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .



In [7]:
# INSERT ... WHERE: a triple-term pattern with variables, matched first, then instantiated into the INSERT template.
g_update.update("""
    INSERT { ?stmt ex:confidence "high" }
    WHERE  { ?stmt rdf:reifies <<( ?s ex:worksFor ?o )>> }
""")
print(g_update.serialize(format="turtle"))

@version "1.2" .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex: <http://example.org/> .
ex:claim ex:confidence "high" ;
	rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .



### 03.01.5.1 Updating a dataset

`update()` can also be used with a dataset.  A dataset holds a default graph and any number of named graphs.  An update that names no graph writes to the default graph. A `GRAPH` block writes to that named graph.  If the named graph does not exists it is created.

To work inside one named graph, `dataset.graph(name)` returns it as a `StarLayerGraph`. Its `update()` stays within that graph: an update that names no graph reads and writes this graph, and an update that names a different graph is refused with a `ValueError`.

In [8]:
from starlayer.graph import StarLayerDataset

# A dataset that knows the prefixes ex and rdf.
dataset = StarLayerDataset(prefixes={"ex": EX, "rdf": RDF})

# The update names no graph, so it goes to the default graph.
dataset.update("""
    INSERT DATA { ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> }
""")

# A GRAPH block names a graph. ex:archive does not exist yet: writing a triple to it creates it.
dataset.update("""
    INSERT DATA { GRAPH ex:archive { ex:claim ex:status ex:retired } }
""")
print("named graphs:", [str(name) for name in dataset.named_graphs()])
print()
print(dataset.serialize(format="trig"))


named graphs: ['<http://example.org/archive>']

@version "1.2" .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex: <http://example.org/> .
ex:archive {
	ex:claim ex:status ex:retired .
}
ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .



In [9]:
# Work inside one named graph. dataset.graph(name) gives it as a StarLayerGraph, here the ex:archive graph created above.
archive = dataset.graph(EX + "archive")

# no GRAPH block: the triple goes in the archive graph, not the default graph
archive.update("INSERT DATA { ex:ann ex:knows ex:bo }")
# DELETE ... WHERE reads and deletes in the archive graph
archive.update("DELETE { ?s ex:knows ?o } INSERT { ?s ex:met ?o } WHERE { ?s ex:knows ?o }")
print(archive.serialize(format="turtle"))

# naming another graph is refused
try:
    archive.update("INSERT DATA { GRAPH ex:other { ex:ann ex:knows ex:cy } }")
except ValueError as error:
    print("refused:", error)

# the dataset has no such limit
dataset.update("INSERT DATA { GRAPH ex:other { ex:ann ex:knows ex:cy } }")
print("named graphs:", sorted(str(name) for name in dataset.named_graphs()))
print(len(list(dataset.quads())), "quads in the dataset;", len(list(archive.quads())), "in the archive graph")

@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix ex: <http://example.org/> .
ex:ann ex:met ex:bo .
ex:claim ex:status ex:retired .

refused: this StarLayerGraph holds graph <http://example.org/archive>; update cannot write to graph <http://example.org/other>
named graphs: ['<http://example.org/archive>', '<http://example.org/other>']
4 quads in the dataset; 2 in the archive graph
